In [2]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen1.5b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

torch: 2.11.0+cu128 cuda: True


In [3]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- "Except X" or "aside from X" followed by negative language about the rest means ONLY X is favored: output [X].
- Ordinal references map A=first, B=second, C=third, D=fourth.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl(PROJECT / "data" / "train.jsonl")
test_raw  = load_jsonl(PROJECT / "data" / "test.jsonl")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])
print("train:", len(train_ds), "test:", len(test_ds))
print("sample:", train_ds[0])

train: 1278 test: 210
sample: {'messages': [{'role': 'system', 'content': 'You are a parser that converts a user\'s natural language response into a subset of the shown options.\n\nThe user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.\n\nOutput format (JSON only, nothing else):\n- A JSON list of the favored labels, e.g. ["A", "B"]\n- [] if the user explicitly rejects ALL options ("none of these", "all wrong")\n- "*" if the utterance is off-topic OR expresses no usable preference ("I don\'t know", "they all look the same", "I love football")\n\nRules:\n- Any positive signal about an option means it goes in the list.\n- "X is better than Y" endorses only X, not Y.\n- "X and Y are both good, X is better" endorses both X and Y.\n- Negations like "not D" or "anything but B" mean the remaining options go in the list.\n- Questions like "is it A?" are treated as tentative endorsem

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print("model loaded. VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

W0811 02:13:46.395000 18080 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


model loaded. VRAM (GB): 1.620461568


In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [6]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [7]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.054575,0.053254,0.058442,422606.000000,0.988829


KeyboardInterrupt: 

In [ ]:
for n, p in model.named_parameters():
    if p.requires_grad:
        print(n, p.dtype)
        break

In [ ]:
from peft import PeftModel

# Base model in 4-bit
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)

# Load adapter
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)

In [ ]:
import gc, torch
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

In [ ]:
def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_cases = [
    ("A and C look good", '["A", "C"]'),
    ("not D", '["A", "B", "C"]'),
    ("I don't know", '"*"'),
    ("none of these work", '[]'),
    ("A is better than B", '["A"]'),
    ("what's for lunch", '"*"'),
    ("hate all of them", '[]'),
    ("A great, B bad, C great, D bad", '["A", "C"]'),
]

for utt, expected in test_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    got = generate_ft(msgs)
    match = "✓" if got.strip() == expected else "✗"
    print(f"{match} {utt!r}\n   expected: {expected}\n   got:      {got}\n")

In [ ]:
from tqdm import tqdm

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

test_examples = load_jsonl(PROJECT / "data" / "test.jsonl")

results = []
for ex in tqdm(test_examples):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(msgs)
    parsed = parse_output(raw)
    correct = parsed is not None and labels_equal(parsed, ex["label"])
    results.append({
        "utterance": ex["utterance"],
        "category": ex.get("category", "unknown"),
        "gold": ex["label"],
        "raw": raw,
        "parsed": parsed,
        "correct": correct,
        "valid_format": parsed is not None,
    })

n = len(results)
n_valid = sum(r["valid_format"] for r in results)
n_correct = sum(r["correct"] for r in results)
print(f"\nformat validity: {n_valid}/{n} = {n_valid/n:.1%}")
print(f"exact match:     {n_correct}/{n} = {n_correct/n:.1%}")

from collections import defaultdict
by_cat = defaultdict(lambda: [0, 0])
for r in results:
    by_cat[r["category"]][0] += 1
    by_cat[r["category"]][1] += int(r["correct"])
print("\nper category:")
for cat, (total, correct) in sorted(by_cat.items()):
    print(f"  {cat}: {correct}/{total} = {correct/total:.1%}")

In [ ]:
print("=== FAILURES ===")
for r in results:
    if not r["correct"]:
        print(f"[{r['category']}] {r['utterance']!r}")
        print(f"   gold:   {r['gold']}")
        print(f"   parsed: {r['parsed']}")
        print(f"   raw:    {r['raw']!r}")
        print()

In [ ]:
import gc, torch
try:
    del base, ft_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

In [ ]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
train = [json.loads(l) for l in (PROJECT / "data" / "train.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
test  = [json.loads(l) for l in (PROJECT / "data" / "test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

print("train negation:", sum(1 for e in train if e.get("category") == "negation"))
print("test  negation:", sum(1 for e in test  if e.get("category") == "negation"))

In [ ]:
neg_train = [e for e in train if e.get("category") == "negation"]

def has_except_all_wrong(u):
    u = u.lower()
    return "except" in u and any(w in u for w in ["all wrong", "all off", "everything is", "everything's", "the rest are wrong", "all are off", "everything else is wrong", "everything else is off"])

def has_ordinal(u):
    u = u.lower()
    ordinals = ["first one", "second one", "third one", "fourth one", "the first", "the second", "the third", "the fourth"]
    return any(o in u for o in ordinals)

print(f"total negation train: {len(neg_train)}")
print(f"  'except X, all wrong'-shape: {sum(1 for e in neg_train if has_except_all_wrong(e['utterance']))}")
print(f"  ordinal references: {sum(1 for e in neg_train if has_ordinal(e['utterance']))}")

print("\nsample 'except X'-ish examples:")
for e in neg_train:
    if "except" in e["utterance"].lower():
        print(f"  {e['utterance']!r} -> {e['label']}")

In [ ]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
synth = [json.loads(l) for l in (PROJECT / "data" / "synthetic_raw.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"total in synthetic_raw: {len(synth)}")
print("by category:", Counter(e.get("category", "unknown") for e in synth))

In [ ]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\parser\data\synthetic_raw.jsonl")

kept, dropped = 0, 0
lines_out = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        if ex.get("category", "unknown") == "unknown":
            dropped += 1
            continue
        lines_out.append(line)
        kept += 1

with open(PATH, "w", encoding="utf-8") as f:
    f.writelines(lines_out)

print(f"kept {kept}, dropped {dropped}")

In [ ]:
import os; print(os.getcwd())

In [ ]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\parser\data\train.jsonl")

except_examples = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        u = ex["utterance"].lower()
        if any(k in u for k in ["except", "aside from", "everyone except", "everything except"]):
            except_examples.append(ex)

print(f"total 'except'-shape examples in train: {len(except_examples)}")
print(f"with single-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) == 1)}")
print(f"with multi-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) > 1)}")
print("\nsamples:")
for e in except_examples[:8]:
    print(f"  {e['utterance']!r} -> {e['label']}")

In [ ]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen3b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROJECT / "data" / "train.jsonl"
TEST_PATH = PROJECT / "data" / "test.jsonl"

In [ ]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_raw = load_jsonl(TRAIN_PATH)
test_raw = load_jsonl(TEST_PATH)
print(f"train: {len(train_raw)}, test: {len(test_raw)}")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]}

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=False, fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
import gc
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from peft import PeftModel

# sanity check what checkpoint dir got saved
print("checkpoints:", os.listdir(OUTPUT_DIR))

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)

# adjust checkpoint number based on what you see above (should be 219 for 1279 train / 16 eff batch * 3 epochs)
CHECKPOINT = "checkpoint-240"
adapter_path = str(OUTPUT_DIR / CHECKPOINT)
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded")

In [ ]:
import os
print(os.listdir(OUTPUT_DIR))

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict

def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

valid, correct = 0, 0
by_cat = defaultdict(lambda: [0, 0])  # [correct, total]
failures = []

for ex in tqdm(test_raw):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    cat = ex.get("category", "unknown")
    by_cat[cat][1] += 1
    if parsed is not None:
        valid += 1
    if parsed is not None and labels_equal(parsed, ex["label"]):
        correct += 1
        by_cat[cat][0] += 1
    else:
        failures.append({"utterance": ex["utterance"], "gold": ex["label"], "parsed": parsed, "raw": raw, "category": cat})

n = len(test_raw)
print(f"\nformat validity: {valid}/{n} = {100*valid/n:.1f}%")
print(f"exact match:     {correct}/{n} = {100*correct/n:.1f}%\n")
print("per category:")
for c in sorted(by_cat):
    ok, tot = by_cat[c]
    print(f"  {c}: {ok}/{tot} = {100*ok/tot:.1f}%")

In [ ]:
print("=== FAILURES ===")
for f in failures:
    print(f"[{f['category']}] {f['utterance']!r}")
    print(f"   gold:   {f['gold']}")
    print(f"   parsed: {f['parsed']}")
    print(f"   raw:    {f['raw']!r}")
    print()

In [ ]:
import json
from pathlib import Path

train = [json.loads(l) for l in open(PROJECT / "data" / "train.jsonl", encoding="utf-8")]
test = [json.loads(l) for l in open(PROJECT / "data" / "test.jsonl", encoding="utf-8")]

train_utts = {e["utterance"].lower().strip() for e in train}
test_utts = {e["utterance"].lower().strip() for e in test}
overlap = train_utts & test_utts
print(f"train: {len(train_utts)}, test: {len(test_utts)}, overlap: {len(overlap)}")
if overlap:
    for u in list(overlap)[:10]:
        print(f"  {u!r}")

In [ ]:
MY_UTTERANCE = "eh, only B feels right"
OPTIONS = ["A", "B", "C", "D"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

In [ ]:
test_utterances = [
    "eh, only B feels right",
    "throw out everything but the third one",
    "hard pass on all four",
    "A? maybe. dunno.",
    "the last two are garbage",
    "B leaves D in the dust",
    "A is as good as C but not better than D",
    "meh whatever pick anything",
    "the second and fourth ones look promising",
    "not really feeling any of them tbh",
    "A is decent, B slightly better, C and D no",
    "give me option C or nothing",
]

for u in test_utterances:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {["A","B","C","D"]}\nUser: {u}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    print(f"{u!r:60s} -> {parsed}")

In [ ]:
MY_UTTERANCE = "E is good"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

In [ ]:
def parse_output(text, valid_letters=None):
    if valid_letters is None:
        valid_letters = ["A","B","C","D"]
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in valid_letters for x in parsed)):
        return parsed
    return None

In [ ]:
tests = [
    ("E is good", ["A","B","C","D","E"]),
    ("A and E are best", ["A","B","C","D","E"]),
    ("everything except E is wrong", ["A","B","C","D","E"]),
    ("the fifth one is good", ["A","B","C","D","E"]),
    ("not E, not D", ["A","B","C","D","E"]),
    ("F is the best", ["A","B","C","D","E","F"]),
    ("the last one only", ["A","B","C","D","E","F","G"]),
    ("none of these", ["A","B","C","D","E","F"]),
    ("I don't know", ["A","B","C","D","E","F"]),
    ("only G matters", ["A","B","C","D","E","F","G"]),
    ("the last one only", ["A","B","C","D","E","F","G","H","I","J"]),
    ("the last one is out for sure", ["A","B","C","D","E","F","G","H","I","J"]),
]

for utt, opts in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {opts}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw, valid_letters=opts)
    print(f"{utt!r:50s} opts={len(opts)} -> raw={raw!r:20s} parsed={parsed}")

In [ ]:
MY_UTTERANCE = "Cant tell really but A is better than the rest"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")